In [1]:
import glob
import os
import re
import csv
import time
import pickle
import networkx as nx
import numpy as np
from optimization_script import run_optimization
from lib.circuitSolver import *
from lib.generate_graph import *
from lib.tableau import *
import matplotlib.pyplot as plt
from path_clustering import path_clustering_lrw
from pathlib import Path

In [2]:
# --- Define Egde reduction function ---
def edge_reduction_func(input_graph):
    our_graph_original = input_graph.copy()
    # Create a mapping dictionary: (0, 0) -> 0, (0, 1) -> 1, ..., (2, 2) -> 8
    mapping = {node: i for i, node in enumerate(our_graph_original.nodes())}

    # Create a new graph with integer labels
    our_graph = nx.relabel_nodes(our_graph_original, mapping)
    graph , circ, statistics = run_optimization(our_graph, edge_reduction=True, minLA=False, opt_CNOT= False)
    return graph

#data analysis functions
def find_saved_graph(node_number, p, idx=None, search_root='.'):
    """Return path to a saved graph matching node_number and p (or None)."""
    p_safe = str(p)
    patterns = [
        f"**/*graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*_graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*p{p_safe}*/**/*graph_n{node_number}_p{p_safe}_idx*.gpickle",
        f"**/*p{p_safe}*/**/*_graph_n{node_number}_p{p_safe}_idx*.gpickle"
    ]
    matches = []
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(search_root, pat), recursive=True))
    matches = sorted(set(matches))
    if not matches:
        return None
    if idx is not None:
        candidates = [m for m in matches if f"_idx{idx}.gpickle" in m]
        if candidates:
            return candidates[0]
    return matches[0]

def load_graph_from_file(path):
    """Load NetworkX graph from a .gpickle or pickled file."""
    try:
        return nx.read_gpickle(path)
    except Exception:
        with open(path, 'rb') as f:
            return pickle.load(f)


In [3]:
# Simulation to obtain the metrics for path clustering algorithm. This will run the optimization for all graphs found in the search path and save the results in a CSV file.
# Where to search for saved graphs (relative to notebook location)
search_root = '.'
# patterns that match both old and new naming conventions
patterns = [
    os.path.join(search_root, '**', 'graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_idx*.gpickle')
]

# gather candidate files
files = []
for pat in patterns:
    files.extend(sorted(glob.glob(pat, recursive=True)))
# filter to .gpickle only and remove duplicates
files = sorted({f for f in files if f.lower().endswith('.gpickle')})
if not files:
    print('No .gpickle files found under', search_root)
else:
    print(f'Found {len(files)} candidate .gpickle files; processing...')

# output CSV path (incrementally appended)
out_csv = os.path.join(search_root, 'graph_stats_path_clustering_fix_bug.csv')
fieldnames = ['path', 'node_number', 'p', 'idx',
              'num_emitters_path', 'num_emitter_CNOTs_path', 'num_gates_path',
              'ordering_path', 'run_time_path']
# ensure header exists (create file if missing)
if not os.path.exists(out_csv):
    with open(out_csv, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

start = time.time()
for i, path in enumerate(files):
    print(f'[{i+1}/{len(files)}] Processing: {path}')
    try:
        # attempt to extract metadata from filename
        m = re.search(r'graph_n(\d+)_p([0-9\.]+)_idx(\d+)', path)
        if m:
            node_n = int(m.group(1))
            p_val = float(m.group(2))
            idx_val = int(m.group(3))
        else:
            m2 = re.search(r'_idx(\d+)\.gpickle', path)
            idx_val = int(m2.group(1)) if m2 else None
            m3 = re.search(r'p([0-9\.]+)', path)
            p_val = float(m3.group(1)) if m3 else None
            m4 = re.search(r'graph_n(\d+)', path)
            node_n = int(m4.group(1)) if m4 else None
        # load graph
        try:
            G = load_graph_from_file(path)
        except NameError:
            G = nx.read_gpickle(path)
        # run optimization
        try:
            t0 = time.time()
            G = edge_reduction_func(G)

            cost, final_ordering = path_clustering_lrw(G, verbose=False)

            ordered_graph = nx.relabel_nodes(G, {old: new for new, old in enumerate(final_ordering)})
            opt_graph, circuit, stats = run_optimization(ordered_graph, edge_reduction=False, minLA=False, opt_CNOT=True)
            run_time = round(time.time() - t0, 7)
        except Exception as e:
            print('run_optimization failed for', path, '->', e)
            stats = None
            final_ordering = None
            run_time = None
        if stats is None:
            num_emitters = None
            num_emitter_CNOTs = None
            num_gates = None
        else:
            num_emitters      = stats[0] if len(stats) > 0 else None
            num_emitter_CNOTs = stats[1] if len(stats) > 1 else None
            num_gates         = stats[2] if len(stats) > 2 else None
        print(f'  -> emitters={num_emitters}, emitter_CNOTs={num_emitter_CNOTs}, gates={num_gates}, run_time={run_time}s')
    except Exception as e:
        print('  ERROR processing', path, ':', e)
        node_n = None
        p_val = None
        idx_val = None
        num_emitters = None
        num_emitter_CNOTs = None
        num_gates = None
        final_ordering = None
        run_time = None
    # write row immediately so results are visible during long runs
    row = {
        'path': path,
        'node_number': node_n,
        'p': p_val,
        'idx': idx_val,
        'num_emitters_path': num_emitters,
        'num_emitter_CNOTs_path': num_emitter_CNOTs,
        'num_gates_path': num_gates,
        'ordering_path': final_ordering,
        'run_time_path': run_time
    }
    # append to CSV (file opened and closed per row to ensure data flushed to disk)
    try:
        with open(out_csv, 'a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writerow(row)
    except Exception as e:
        print('  ERROR writing row to CSV:', e)

elapsed = time.time() - start
print(f'Done. Processed {len(files)} files in {elapsed:.1f}s. Results saved (incrementally) to {out_csv}')


Found 3000 candidate .gpickle files; processing...
[1/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx1.gpickle
  -> emitters=1, emitter_CNOTs=0, gates=11, run_time=2.6250119s
[2/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx10.gpickle
  -> emitters=1, emitter_CNOTs=0, gates=11, run_time=2.1296761s
[3/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx2.gpickle
  -> emitters=2, emitter_CNOTs=2, gates=16, run_time=2.1785879s
[4/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx3.gpickle
  -> emitters=1, emitter_CNOTs=0, gates=11, run_time=2.116729s
[5/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx4.gpickle
  -> emitters=1, emitter_CNOTs=0, gates=11, run_time=2.1825628s
[6/3000] Processing: ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx5.gpickle
  -> emitters=2, emitter_CNOTs=1, gates=13, run_time=2.1542

KeyboardInterrupt: 

In [5]:
# Simulation to obtain the metrics for path clustering algorithm. 
# This will run the optimization for all graphs found in the search path and save the results in a CSV file.
# Where to search for saved graphs (relative to notebook location)
search_root = '.'
# patterns that match both old and new naming conventions
patterns = [
    os.path.join(search_root, '**', 'graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*graph_n*_p*_idx*.gpickle'),
    os.path.join(search_root, '**', '*_idx*.gpickle')
]

# gather candidate files
files = []
for pat in patterns:
    files.extend(sorted(glob.glob(pat, recursive=True)))
# filter to .gpickle only and remove duplicates
files = sorted({f for f in files if f.lower().endswith('.gpickle')})
if not files:
    print('No .gpickle files found under', search_root)
else:
    print(f'Found {len(files)} candidate .gpickle files; processing...')

# output CSV path (incrementally appended)
out_csv = os.path.join(search_root, 'graph_stats_path_clustering_fix_bug.csv')
fieldnames = ['path', 'node_number', 'p', 'idx',
              'num_emitters_path', 'num_emitter_CNOTs_path', 'num_gates_path',
              'ordering_path', 'run_time_path']
# ensure header exists (create file if missing)
if not os.path.exists(out_csv):
    with open(out_csv, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

# ── Resume support: skip paths already written to out_csv ──────────────────
done_paths = set()
if os.path.exists(out_csv):
    with open(out_csv, 'r', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if row.get('path'):
                done_paths.add(row['path'])
if done_paths:
    print(f'Resuming: {len(done_paths)} file(s) already in {out_csv}, will be skipped.')

start = time.time()
for i, path in enumerate(files):
    if path in done_paths:
        print(f'[{i+1}/{len(files)}] Skipping (already done): {path}')
        continue
    print(f'[{i+1}/{len(files)}] Processing: {path}')
    try:
        # attempt to extract metadata from filename
        m = re.search(r'graph_n(\d+)_p([0-9\.]+)_idx(\d+)', path)
        if m:
            node_n = int(m.group(1))
            p_val = float(m.group(2))
            idx_val = int(m.group(3))
        else:
            m2 = re.search(r'_idx(\d+)\.gpickle', path)
            idx_val = int(m2.group(1)) if m2 else None
            m3 = re.search(r'p([0-9\.]+)', path)
            p_val = float(m3.group(1)) if m3 else None
            m4 = re.search(r'graph_n(\d+)', path)
            node_n = int(m4.group(1)) if m4 else None
        # load graph
        try:
            G = load_graph_from_file(path)
        except NameError:
            G = nx.read_gpickle(path)
        # run optimization
        try:
            t0 = time.time()
            G = edge_reduction_func(G)

            cost, final_ordering = path_clustering_lrw(G, verbose=False)

            ordered_graph = nx.relabel_nodes(G, {old: new for new, old in enumerate(final_ordering)})
            opt_graph, circuit, stats = run_optimization(ordered_graph, edge_reduction=False, minLA=False, opt_CNOT=True)
            run_time = round(time.time() - t0, 7)
        except Exception as e:
            print('run_optimization failed for', path, '->', e)
            stats = None
            final_ordering = None
            run_time = None
        if stats is None:
            num_emitters = None
            num_emitter_CNOTs = None
            num_gates = None
        else:
            num_emitters      = stats[0] if len(stats) > 0 else None
            num_emitter_CNOTs = stats[1] if len(stats) > 1 else None
            num_gates         = stats[2] if len(stats) > 2 else None
        print(f'  -> emitters={num_emitters}, emitter_CNOTs={num_emitter_CNOTs}, gates={num_gates}, run_time={run_time}s')
    except Exception as e:
        print('  ERROR processing', path, ':', e)
        node_n = None
        p_val = None
        idx_val = None
        num_emitters = None
        num_emitter_CNOTs = None
        num_gates = None
        final_ordering = None
        run_time = None
    # write row immediately so results are visible during long runs
    row = {
        'path': path,
        'node_number': node_n,
        'p': p_val,
        'idx': idx_val,
        'num_emitters_path': num_emitters,
        'num_emitter_CNOTs_path': num_emitter_CNOTs,
        'num_gates_path': num_gates,
        'ordering_path': final_ordering,
        'run_time_path': run_time
    }
    # append to CSV (file opened and closed per row to ensure data flushed to disk)
    try:
        with open(out_csv, 'a', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writerow(row)
    except Exception as e:
        print('  ERROR writing row to CSV:', e)

elapsed = time.time() - start
print(f'Done. Processed {len(files)} files in {elapsed:.1f}s. Results saved (incrementally) to {out_csv}')


Found 3000 candidate .gpickle files; processing...
Resuming: 1850 file(s) already in ./graph_stats_path_clustering_fix_bug.csv, will be skipped.
[1/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx1.gpickle
[2/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx10.gpickle
[3/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx2.gpickle
[4/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx3.gpickle
[5/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx4.gpickle
[6/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx5.gpickle
[7/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx6.gpickle
[8/3000] Skipping (already done): ./random_graph_db/graph_db_node_10/01_p0.1/01_graph_n10_p0.1_idx